# Fair product-line search: RFC with common random numbers vs `orthant_cdf`

Section 7 of `rfc_vs_exact.ipynb` compared exact scoring with RFC at
$R = 250{,}000$, which is the number of draws needed for 0.1-point accuracy
on every share. To rank lines, RFC only needs the differences between lines
to be right. This notebook gives RFC that advantage: one set of $R$ draws,
shared by every line (common random numbers), at $R = 10^2, 10^3, 10^4$.

**Setup: the same as section 7.**

- 720 candidate products, lines of 3, the same 3 competitors ($K = 6$, 5-dimensional orthants).
- The same margin objective.
- The aggregate model: true $\beta$, $\Sigma_A$, $\sigma_P$.

Section 7 has no respondents or posterior means, so none are used here.
Scoring $10^6$ lines over 1,000 respondents would take about $3 \cdot 10^9$
orthants, far beyond the 30-minute cap.

**Differences from section 7**

- Section 7 did not save its random lines, so $10^6$ lines are drawn again
  from a fixed seed. Section 7's best line, (279, 454, 459) from the greedy
  search, is added to them.
- Shares are renormalized `orthant_cdf`: all 6 shares of each line are
  computed and divided by their sum (see `orthant_accuracy.ipynb`). Lines
  where any difference covariance reaches a correlation of 0.9 or more are
  flagged, because there the error is larger (sd 0.0065, max 0.034 at
  $d = 4$).
- No SciPy.

**Reference ranking**: renormalized `orthant_cdf` profit. The profit gap of
each RFC pick is also measured with `orthant_cdf`.

**Runtime**: every configuration ran on all lines. The timing check put the
total at about 15 minutes, so no subset was needed.

In [1]:
import json, time
import numpy as np, pandas as pd
from scipy import stats
import rfc
pd.set_option("display.precision", 4)

lev_all, price_all, X_all = rfc.all_products()
comp_lev = np.array([[1, 2, 0, 0, 1], [3, 2, 2, 0, 0], [1, 1, 2, 1, 0]])
comp_price = np.array([-1.0, 0.5, -0.5])
comp_idx = np.array([np.flatnonzero((lev_all == l).all(1) & (price_all == p))[0] for l, p in zip(comp_lev, comp_price)])
X_comp = X_all[comp_idx]
print("competitors (section 7):", comp_idx.tolist(), " margins", rfc.dollars(X_comp))

rng = np.random.default_rng(2026)
lines = rfc.random_lines(1_000_000, len(X_all), 3, rng)
sec7_best = np.array([279, 454, 459])
if not (lines == sec7_best).all(1).any():
    lines[-1] = sec7_best
print(f"{len(lines):,} distinct lines; section 7 best line included")

competitors (section 7): [305, 703, 291]  margins [-2.4  6.4 -0.5]


1,000,000 distinct lines; section 7 best line included


In [2]:
t = time.perf_counter()
ref, ref_sh, ref_corr = rfc.line_profit_orthant_renorm(X_all[lines], X_comp)
t_orth = time.perf_counter() - t
flag = ref_corr >= 0.9
order = np.argsort(-ref)
best = order[0]
print(f"orthant_cdf (renormalized): {len(lines):,} lines, {6 * len(lines):,} orthants in {t_orth:.1f} s")
print(f"lines flagged (max difference-covariance corr >= 0.9): {flag.sum():,} of {len(lines):,}; "
      f"in the top 10: {flag[order[:10]].sum()}, top 100: {flag[order[:100]].sum()}, top 1,000: {flag[order[:1000]].sum()}")
print("reference best line:", lines[best].tolist(), f"profit {ref[best]:.4f}", "(flagged)" if flag[best] else "")
for i in lines[best]:
    print("   levels", lev_all[i].tolist(), f"price {price_all[i]:+.1f}", f"margin ${rfc.dollars(X_all[i]):.2f}")
print(f"profit spread: top 10 {ref[order[0]] - ref[order[9]]:.4f}, top 100 {ref[order[0]] - ref[order[99]]:.4f}")

orthant_cdf (renormalized): 1,000,000 lines, 6,000,000 orthants in 41.1 s
lines flagged (max difference-covariance corr >= 0.9): 1,296 of 1,000,000; in the top 10: 3, top 100: 7, top 1,000: 38
reference best line: [279, 454, 459] profit 0.6921 (flagged)
   levels [1, 1, 1, 1, 1] price +1.0 margin $3.70
   levels [2, 1, 1, 1, 0] price +1.0 margin $4.70
   levels [2, 1, 1, 1, 1] price +1.0 margin $4.50
profit spread: top 10 0.0743, top 100 0.1116


## RFC with common random numbers

Each $R$ uses one fixed draw of $(E_A, E_P)$ for all $10^6$ lines (RFC variant
a, the same error model as the exact shares). Wall times are measured on the
full set of lines. Spearman correlation is computed on the reference's top
1,000 lines.

In [3]:
rows, picks = [], {}
for R in (100, 1_000, 10_000):
    t = time.perf_counter()
    p_rfc, _ = rfc.line_profit_rfc_crn(X_all[lines], X_comp, R, np.random.default_rng(R))
    dt = time.perf_counter() - t
    o = np.argsort(-p_rfc, kind="stable")
    picks[R] = o[0]
    rows.append(dict(R=R, seconds=dt, same_best=bool(o[0] == best),
                     top10_overlap=len(np.intersect1d(o[:10], order[:10])),
                     top100_overlap=len(np.intersect1d(o[:100], order[:100])),
                     spearman_top1000=stats.spearmanr(ref[order[:1000]], p_rfc[order[:1000]]).statistic,
                     pick=lines[o[0]].tolist(), pick_ref_rank=int(np.flatnonzero(order == o[0])[0]) + 1,
                     profit_gap=ref[best] - ref[o[0]], pick_flagged=bool(flag[o[0]])))
rows.append(dict(R="orthant_cdf (reference)", seconds=t_orth, same_best=True, top10_overlap=10, top100_overlap=100,
                 spearman_top1000=1.0, pick=lines[best].tolist(), pick_ref_rank=1,
                 profit_gap=0.0, pick_flagged=bool(flag[best])))
fair = pd.DataFrame(rows).set_index("R")
fair

,seconds,same_best,top10_overlap,top100_overlap,spearman_top1000,pick,pick_ref_rank,profit_gap,pick_flagged
R,,,,,,,,,
100,5.1255,False,0,5,0.2887,"[74, 264, 634]",25648,0.3068,False
1000,46.2416,True,5,67,0.6340,"[279, 454, 459]",1,0.0000,True
10000,435.5974,True,8,84,0.8218,"[279, 454, 459]",1,0.0000,True
orthant_cdf (reference),41.1249,True,10,100,1.0000,"[279, 454, 459]",1,0.0000,True


In [4]:
ok = fair.iloc[:-1]
ok = ok[ok.same_best & (ok.top10_overlap >= 9)]
if len(ok):
    R_min = int(ok.index[0]); t_min = float(ok.seconds.iloc[0])
    verdict = (f"smallest R with the same best line and top-10 overlap >= 9: R = {R_min:,}, "
               f"{t_min:.0f} s; orthant_cdf took {t_orth:.0f} s, speedup {t_min / t_orth:.2f}x")
else:
    R_min, t_min = None, None
    verdict = f"no R up to 10,000 picks the same best line with top-10 overlap >= 9; orthant_cdf took {t_orth:.0f} s"
print(verdict)
json.dump(dict(R_min=R_min, t_rfc=t_min, t_orth=t_orth, n_lines=len(lines), verdict=verdict,
               table=json.loads(fair.reset_index().to_json(orient="records"))),
          open("product_line_fair.json", "w"), indent=1)

no R up to 10,000 picks the same best line with top-10 overlap >= 9; orthant_cdf took 41 s


**Verdict: no R up to 10⁴ meets the bar.** None picks the same best line and
also gets at least 9 of the reference's top 10. `orthant_cdf` scored all 10⁶
lines in 41 s.

- **R = 100 draws** (5 s) is useless for this. Its pick ranks 25,648th, with a profit
  gap of 0.31.
- **R = 1,000 draws** (46 s, about the same time as `orthant_cdf`) already picks the
  same best line, (279, 454, 459), so its profit gap is 0. It gets only 5 of
  the top 10 (Spearman 0.63 on the top 1,000).
- **R = 10,000 draws** (436 s, 10.6 times as long as `orthant_cdf`'s 41 s) also picks the same
  best line. It gets 8 of the top 10 and 84 of the top 100 (Spearman 0.82).
  The bar therefore needs more than 10,000 draws, so any RFC setting that
  meets it takes more than 10.6 times as long as `orthant_cdf`.

**Flag.** The best line is flagged: 454 and 459 differ only on A5, and its
difference covariances reach a correlation of 0.9 or more. So are 3 of the top
10 lines, 7 of the top 100 and 38 of the top 1,000. That is the regime where
renormalized `orthant_cdf` has share sd 0.0065 and max error 0.034 at
$d = 4$, so top-10 overlap is measured against a reference that is itself
uncertain there. In section 7, SciPy agreed with `orthant_cdf` on this winner
and on the top six lines.